# L-Bracket Bolted Connection — Bolt Group Analysis

Determine bolt reaction forces for an L-bracket bolted to a wall plate with 2x M8 bolts under a 500N vertical load at the free end.

## Assumptions
- Rigid bracket (no flexural deformation in bolt group analysis)
- Bolts carry only shear (no tension/prying)
- Equal stiffness all bolts
- Load applied at free end centroid
- No friction between bracket and wall plate
- Room temperature, no thermal effects

In [1]:
import sympy as sp

v, n, e, r = sp.symbols('V n e r', positive=True)

# Direct shear per bolt
v_direct = v / n

# Moment about bolt group centroid
m = v * e

# Moment-induced shear per bolt (tangential, perpendicular to radius)
v_moment = m / (n * r)

# Critical bolt resultant (direct and moment shears are perpendicular)
f_bolt = sp.sqrt(v_direct**2 + v_moment**2)

print('Direct shear per bolt:', v_direct)
print('Moment-induced shear per bolt:', v_moment)
print('Critical bolt resultant:', sp.simplify(f_bolt))

Direct shear per bolt: V/n
Moment-induced shear per bolt: V*e/(n*r)
Critical bolt resultant: V*sqrt(e**2 + r**2)/(n*r)


In [2]:
import pint
import numpy as np

ureg = pint.UnitRegistry()

# Bracket geometry
vert_height = 80.0 * ureg.mm
horiz_length = 60.0 * ureg.mm
thickness = 5.0 * ureg.mm
bolt_spacing = 50.0 * ureg.mm
num_bolts = 2

# Material: ASTM A36, Roark's 8th ed. Table A.5
yield_stress = 250.0 * ureg.MPa

# Loading
applied_load = 500.0 * ureg.N

# Derived
eccentricity = (horiz_length + thickness / 2).to(ureg.mm)
bolt_group_radius = (bolt_spacing / 2).to(ureg.mm)

print(f'Eccentricity: {eccentricity}')
print(f'Bolt group radius: {bolt_group_radius}')

Eccentricity: 62.5 millimeter
Bolt group radius: 25.0 millimeter


In [3]:
# Bolt group analysis
direct_shear = (applied_load / num_bolts).to(ureg.N)
moment_shear = (
    applied_load * eccentricity / (num_bolts * bolt_group_radius)
).to(ureg.N)

critical_resultant = (
    np.sqrt(direct_shear.magnitude**2 + moment_shear.magnitude**2) * ureg.N
)

print(f'Direct shear per bolt: {direct_shear}')
print(f'Moment shear per bolt: {moment_shear}')
print(f'Critical bolt resultant: {critical_resultant:.1f}')

Direct shear per bolt: 250.0 newton
Moment shear per bolt: 625.0 newton
Critical bolt resultant: 673.1 newton


## Expected Values

| Parameter | Value | Note |
|-----------|-------|------|
| Direct shear per bolt | 250 N | V/n |
| Moment shear per bolt | 625 N | V*e/(n*r) |
| Critical bolt resultant | 673 N | sqrt(250^2 + 625^2) |
| Bracket stress | < 250 MPa | Below A36 yield |